In [1]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from torch import optim
from torch.utils.data import Subset
from model import DayCentModel
from data import DayCentDataset
import os
import wandb
from utils import evaluate

In [2]:
# Reproducibility
RND_SEED = 42
np.random.seed(RND_SEED)
torch.manual_seed(RND_SEED)

In [14]:
EXPERIMENT_ID = "experiment10"
INPUT_NPY = f"/users/6/mehta423/projects/daycent/data/{EXPERIMENT_ID}/train_50_X.npy"
OUTPUT_NPY = f"/users/6/mehta423/projects/daycent/data/{EXPERIMENT_ID}/train_50_Y.npy"
INIT_COND = "/users/6/mehta423/projects/daycent/data/SAS_KGML_090925/InputData/initial_site_conditions.xlsx"
OUTPUT_DIR = f"/users/6/mehta423/projects/daycent/output/{EXPERIMENT_ID}"

In [15]:
# ----------------------
# Config
# ----------------------
BATCH_SIZE = 2048
EPOCHS = 100
LR = 1e-2
DEVICE = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")

run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    # Set the wandb project where this run will be logged.
    project="daycent",
    name="experiment10.2/pid-holdout-1quadrant-50scenarios",
    notes="For this the holdout is quadrant wise. The right side is for testing and the left side is for training. Using 50 scenarios.",
    config={
        "learning_rate": LR,
        "architecture": "LSTM with Attention",
        "dataset": "50 Scenarios, 55 Points",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE
    },
)

In [16]:
# ----------------------
# Dataloader
# ----------------------
# check dataset
dataset = DayCentDataset(INPUT_NPY, OUTPUT_NPY, INIT_COND, apply_scaling=True)

0


In [17]:
len(dataset)

70000

In [23]:
NUM_SCENARIOS = 50
UNIT_SIZE = int(len(dataset) / NUM_SCENARIOS)
train_size = int(UNIT_SIZE * (NUM_SCENARIOS * 0.7))
val_size = int(UNIT_SIZE * (NUM_SCENARIOS * 0.2))
test_size = int(UNIT_SIZE * (NUM_SCENARIOS * 0.1))
# Total: 25000

# 2) Create index arrays for each split
train_idx = np.arange(0, train_size)
val_idx = np.arange(train_size, train_size + val_size)
test_idx = np.arange(train_size + val_size, train_size + val_size + test_size)

# 3) Wrap subsets
train_ds = Subset(dataset, train_idx)
val_ds   = Subset(dataset, val_idx)
test_ds  = Subset(dataset, test_idx)

# 4) Create loaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Dataset sizes — total: {len(dataset)}, train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}")

Dataset sizes — total: 70000, train: 49000, val: 14000, test: 7000


In [24]:
# infer input dim
sample = dataset[0]
seq_feat_dim = sample["sequence"].shape[1]  # #features
init_dim = sample["init_cond"].shape[0]
year_dim = sample["year_enc"].shape[0]

print(f"Input feature dim: {seq_feat_dim}, init cond dim: {init_dim}, year enc dim: {year_dim}")

Input feature dim: 20, init cond dim: 245, year enc dim: 16


In [25]:

model = DayCentModel(input_dim=seq_feat_dim, init_dim=init_dim, year_dim=year_dim)
model.to(DEVICE)

DayCentModel(
  (init_proj): Linear(in_features=261, out_features=32, bias=True)
  (daily_proj): Linear(in_features=20, out_features=32, bias=True)
  (lstm): LSTM(64, 128, num_layers=2, batch_first=True)
  (somsc_attn): AttentionPooling(
    (attn): Linear(in_features=128, out_features=1, bias=True)
    (proj): Linear(in_features=128, out_features=128, bias=True)
  )
  (yield_attn): AttentionPooling(
    (attn): Linear(in_features=128, out_features=1, bias=True)
    (proj): Linear(in_features=128, out_features=128, bias=True)
  )
  (somsc_head): Linear(in_features=128, out_features=1, bias=True)
  (yield_head): Linear(in_features=128, out_features=1, bias=True)
)

In [26]:
# ----------------------
# Optimizer & scheduler
# ----------------------
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5)

In [27]:
# ----------------------
# Training loop with loss tracking
# ----------------------
best_val_loss = float('inf')


# Initialize loss tracking lists
train_losses = []
val_somsc_losses = []
val_yield_losses = []
val_total_losses = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        # move to device
        batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        optimizer.zero_grad()
        out = model(batch)

        # --- SOMSC loss ---
        somsc_target = batch["somsc"]              # (B,12)
        somsc_mask = batch["somsc_mask"]           # (B,12)

        # compute masked MSE
        somsc_loss = ((out["somsc_pred"].squeeze(-1) - somsc_target)**2 * somsc_mask).sum() / somsc_mask.sum()

        # --- Yield loss ---
        yield_target = batch["yield"]              # (B,)
        yield_mask = batch["yield_mask"]           # (B,)
        yield_loss = ((out["yield_pred"] - yield_target)**2 * yield_mask).sum() / yield_mask.sum()

        # --- total loss ---
        alpha = 1.0  # weight for SOMSC loss
        beta = 1.0   # weight for Yield loss
        loss = alpha*somsc_loss + beta*yield_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch["sequence"].size(0)

    total_loss /= len(dataset)
    train_losses.append(total_loss)
    
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss:.4f}")
    somsc_loss_val, yield_loss_val = evaluate(model, val_loader, DEVICE)
    print(f"  Val SOMSC Loss: {somsc_loss_val:.4f}, Yield Loss: {yield_loss_val:.4f}")

    if yield_loss_val < best_val_loss:
        best_val_loss = yield_loss_val
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_model.pth"))
        print(f"Saved best model at epoch {epoch} with val_loss: {yield_loss_val:.4f}")
    
    # Track validation losses
    val_somsc_losses.append(somsc_loss_val)
    val_yield_losses.append(yield_loss_val)
    val_total_losses.append(somsc_loss_val + yield_loss_val)

    run.log({
        "epoch": epoch + 1,
        "train_loss": total_loss,
        "val_somsc_loss": somsc_loss_val,
        "val_yield_loss": yield_loss_val,
        "val_total_loss": somsc_loss_val + yield_loss_val,
        "learning_rate": optimizer.param_groups[0]['lr'],
    })

    scheduler.step(total_loss)


Epoch 1/100 - Loss: 0.9659
  Val SOMSC Loss: 0.1378, Yield Loss: 0.8834
Saved best model at epoch 0 with val_loss: 0.8834
Epoch 2/100 - Loss: 0.7208
  Val SOMSC Loss: 0.1116, Yield Loss: 0.8008
Saved best model at epoch 1 with val_loss: 0.8008
Epoch 3/100 - Loss: 0.6524
  Val SOMSC Loss: 0.1183, Yield Loss: 0.8581
Epoch 4/100 - Loss: 0.6248
  Val SOMSC Loss: 0.1213, Yield Loss: 0.7723
Saved best model at epoch 3 with val_loss: 0.7723
Epoch 5/100 - Loss: 0.6152
  Val SOMSC Loss: 0.1312, Yield Loss: 0.8601
Epoch 6/100 - Loss: 0.6690
  Val SOMSC Loss: 0.1201, Yield Loss: 0.8452
Epoch 7/100 - Loss: 0.6942
  Val SOMSC Loss: 0.1126, Yield Loss: 0.7888
Epoch 8/100 - Loss: 0.6470
  Val SOMSC Loss: 0.1157, Yield Loss: 0.7852
Epoch 9/100 - Loss: 0.6066
  Val SOMSC Loss: 0.1138, Yield Loss: 0.7121
Saved best model at epoch 8 with val_loss: 0.7121
Epoch 10/100 - Loss: 0.6483
  Val SOMSC Loss: 0.1192, Yield Loss: 0.4935
Saved best model at epoch 9 with val_loss: 0.4935
Epoch 11/100 - Loss: 0.5789
 

In [12]:
evaluate(model, test_loader, DEVICE)

(0.10696528736282797, 0.11560259650735294)

In [13]:
run.finish()


epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▆▆▆▆▆▇▇▇▇▇▇████
learning_rate,█████████████████▄▄▄▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,█▇▆▆▆▆▆▅▆▃▁▁▁▁▁▂▇▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_somsc_loss,█▆▄▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▅▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_total_loss,██▇▇▇▇█▇▆▃▂▁▃▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_yield_loss,█▇▇▆▇▆▇▆▅▃▂▁▃▁▁▁▁▂▁█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,100
learning_rate,2e-05
train_loss,0.09517
val_somsc_loss,0.03887
val_total_loss,0.1724
